| Event                                  |       Date |
|----------------------------------------|-----------:|
| Start of first lockdown                | 2020-03-24 |
| Easing phase one                       | 2020-05-29 |
| Easing phase two                       | 2020-06-19 |
| Easing phase three                     | 2020-07-10 |
| Five-tier restriction                  | 2020-11-02 |
| Start of second lockdown — mainland    | 2021-01-05 |
| Start of vaccination                   | 2021-01-25 |
| Start of Euro 2020 football tournament | 2021-06-11 |
| End of Euro 2020 football tournament   | 2021-07-11 |
| Start of COP26                         | 2021-10-31 |
| End of COP26                           | 2021-11-11 |

In [6]:
from scipy.special import logit
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

import numpy as np

from utils import data

In [2]:
long_pd = data.load_individual_features(format="long")
long_pd = long_pd.to_pandas()

In [3]:
long_pd["seq_window"] = (long_pd["sequence_id"].astype(str)
                     + "_"
                     + long_pd["window_id"].astype(str))

long_pd["seq_window"] = long_pd["seq_window"].astype(dtype="category")

long_pd["resolution_z"]  = (
        long_pd["resolution"]-long_pd["resolution"].mean()
)/long_pd["resolution"].std()

long_pd["logit_seq_prop"] = logit(long_pd["wn_prop_sequenced"])

long_pd["logit_seq_prop_z"] = (
    long_pd["logit_seq_prop"] - long_pd["logit_seq_prop"].mean()
) / long_pd["logit_seq_prop"].std()

In [ ]:
agg = long_pd.groupby(["window_id", "sequence_id"], observed=True).agg(
    success = ("in_non_singleton", "sum"),
    trails = ("in_non_singleton", "count"),
    n_seq_windows = ("seq_window", "nunique"),
    mean_resolution = ("resolution", "mean"),
    mean_logit_seq_prop = ("logit_seq_prop", "mean")
)

In [14]:
long_pd.to_parquet("R/data/cluster_data.parquet", index=False)

In [4]:
long_pd.columns

Index(['patient_id', 'sequence_id', 'resolution', 'window_id', 'window_idx',
       'wn_mid_date', 'collection_date', 'wn_prop_sequenced', 'age_band',
       'age_group', 'is_female', 'age_midpoint', 'is_vaccinated',
       'pango_lineage', 'who_voc', 'datazone', 'dz_simd_quintile',
       'dz_simd_decile', 'overall_zscore', 'income_zscore',
       'employment_zscore', 'education_zscore', 'health_zscore',
       'access_zscore', 'crime_zscore', 'housing_zscore', 'in_non_singleton',
       'wave', 'seq_window', 'resolution_z', 'logit_seq_prop',
       'logit_seq_prop_z'],
      dtype='object')

In [ ]:
waves = [
    'WV1_B.1.177_C108360',
    'WV2_B.1.1.7_C574152',
    'WV3_AY.4_C983568',
     'WV4_BA.2_C479360',
    'WV5_BA.2_C85080',
    'WV6_BA.5.2_C23016'
]

In [ ]:
formula = """
    in_non_singleton ~
        C(dz_simd_quintile, Treatment(3))
        + C(age_group, Treatment('40–59'))
        + is_female
        + resolution_z
        + I(resolution_z ** 2)
        + logit_seq_prop_z
        + resolution_z:logit_seq_prop_z
"""
        # + bs(overall_zscore, df=4)

wave_models = {}
for wave, wave_df in long_pd.groupby("wave", observed=True):
    wave_df["seq_window"] = wave_df["seq_window"].cat.remove_unused_categories()
    model = smf.gee(
        formula,
        groups="seq_window",
        data=wave_df,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    )
    wave_models[wave] = model.fit()
    print(f"\n=== Wave {wave} ===")
    print(wave_models[wave].summary())

In [ ]:
def extract(name, res):
    qic, qicu = res.qic(scale=1)
    ci = res.conf_int()
    return pd.DataFrame({
        "wave":     name,
        "aic":       res.aic,
        "qic":       qic,
        "qicu":      qicu,
        "rho":       res.cov_struct.dep_params,
        "scale":     res.scale,
        "nobs":      res.nobs,
        "converged": res.converged,
        "term":      res.params.index,
        "coef":      res.params.values,
        "se":        res.bse.values,
        "z":         res.tvalues.values,
        "pvalue":    res.pvalues.values,
        "ci_lo":     ci.iloc[:, 0].values,
        "ci_hi":     ci.iloc[:, 1].values,
    })

results = pd.concat([extract(name, res) for name, res in wave_models.items()], ignore_index=True)
results.to_csv("regression_per_wave_results_adj_resolution.csv", index=False)

In [ ]:
"""
Grouped sentences into temporal windows and for each window clustered then using Leiden community detection at multiple resolutions (0.1-0.8). Increasing resolution produces finer clusters. I fitted a regression model to look at associations with demographic and socioeconomic deprivation. I fitted data from multiple waves of the epidemic. Help me analyse the results and write a report. The main outcome variable is whether a sentence is in a non-singleton cluster (i.e. a cluster with more than one sentence). The main predictors are age group, sex, socioeconomic deprivation (measured by the Scottish Index of Multiple Deprivation, SIMD), and resolution. I also included an interaction term between resolution and the proportion of sentences sequenced in that window. I used a Generalized Estimating Equations (GEE) model with an exchangeable correlation structure to account for clustering within sequence-window groups. The results are presented in the attached CSV file, which includes coefficients, standard errors, z-values, p-values, confidence intervals, and model fit statistics for each wave of the epidemic.
"""